# C08 — Contrastive Self-Supervised Learning: SimCLR, CLIP, and DINOv2

> **Audience**: PhD students · **Framework**: PyTorch + Hugging Face · **Dataset**: STL-10 (unlabelled)

Self-supervised learning (SSL) trains representations from unlabelled data —
the dominant paradigm for foundation model pretraining.

**Progression in this notebook**
```
SimCLR  → contrastive SSL using augmentation pairs (vision only)
    ↓
CLIP    → contrastive SSL across vision and language modalities
    ↓
DINOv2  → distillation-based SSL; best general-purpose visual features today
```

**Why self-supervised?**
Labels are expensive and scarce. SSL extracts supervisory signal from the
structure of data itself: augmentation invariance (SimCLR), cross-modal
alignment (CLIP), or self-distillation (DINO/DINOv2).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Device: {DEVICE}")

# 1) SimCLR — Simple Contrastive Learning of Representations

**Core idea** (Chen et al., 2020)

Given an unlabelled image $x$, apply two *different* random augmentations
to get a positive pair $(\tilde{x}_i, \tilde{x}_j)$.
All other images in the batch form negatives.

Train an encoder + projection head so that:
- Augmentation pairs (positives) are pulled together in embedding space
- All other pairs in the batch (negatives) are pushed apart

**NT-Xent loss** (Normalised Temperature-scaled Cross-Entropy)

$$L_i = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

- $z_i, z_j$: L2-normalised projection head outputs for the augmentation pair
- $\tau$: temperature hyperparameter — lower = harder, sharper distribution
- The denominator sums over $2N-2$ negatives (the rest of the batch)

**Why a projection head?**
Chen et al. found empirically that the representations one layer *before* the
projection head (the backbone output) transfer better to downstream tasks.
The projection head is discarded after SSL pretraining.

In [ ]:
class SimCLRAugmentation(nn.Module):
    """
    Two-view augmentation pipeline for SimCLR.

    Applies two independent random augmentations to the same image,
    producing a positive pair (x_i, x_j).

    Design choices following Chen et al.:
    - RandomResizedCrop: most important augmentation for SimCLR
    - ColorJitter: prevents the model from using colour shortcuts
    - RandomGrayscale: further breaks colour reliance
    - GaussianBlur: important for high-resolution images; less so for 32×32
    """

    def __init__(self, img_size: int = 32, s: float = 0.5) -> None:
        super().__init__()
        colour_jitter = transforms.ColorJitter(
            brightness=0.8*s, contrast=0.8*s, saturation=0.8*s, hue=0.2*s
        )
        self.augment = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([colour_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __call__(self, x) -> tuple:
        return self.augment(x), self.augment(x)


class SimCLRDataset(Dataset):
    """Wraps an image dataset to return augmentation pairs instead of (image, label)."""

    def __init__(self, base_dataset, augmentation: SimCLRAugmentation) -> None:
        self.base       = base_dataset
        self.augment    = augmentation

    def __len__(self): return len(self.base)

    def __getitem__(self, idx) -> tuple:
        img, _ = self.base[idx]  # discard label — this is self-supervised
        xi, xj = self.augment(img)
        return xi, xj


# Build STL-10 unlabelled dataset (100k images, no labels, 96×96)
stl10_base = torchvision.datasets.STL10(
    "./data", split="unlabeled", download=True,
    transform=transforms.ToPILImage()  # keep as PIL for augmentation pipeline
)

augmentor      = SimCLRAugmentation(img_size=96)
simclr_dataset = SimCLRDataset(stl10_base, augmentor)
simclr_loader  = DataLoader(simclr_dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

xi_sample, xj_sample = simclr_dataset[0]
print(f"STL-10 unlabelled size : {len(stl10_base)}")
print(f"Augmented pair shapes  : {xi_sample.shape}, {xj_sample.shape}")

In [ ]:
class ProjectionHead(nn.Module):
    """
    Non-linear projection head for SimCLR.

    Maps backbone features to a lower-dimensional space where the
    contrastive loss is applied. Discarded after SSL pretraining.

    Architecture: Linear → BN → ReLU → Linear
    """

    def __init__(self, in_dim: int, hidden_dim: int = 512, out_dim: int = 128) -> None:
        super().__init__()

        # (batch_num, in_dim) → (batch_num, out_dim)
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, in_dim) → (batch_num, out_dim)
        z = self.proj(x)
        # L2 normalise — contrastive loss operates on unit sphere
        # (batch_num, out_dim) → (batch_num, out_dim)
        return F.normalize(z, p=2, dim=1)


class NTXentLoss(nn.Module):
    """
    Normalised Temperature-scaled Cross-Entropy loss for SimCLR.

    Given a batch of N pairs, treats the 2N embeddings as a single set.
    For each embedding z_i, the loss maximises similarity to its augmentation
    partner z_j relative to all 2N-2 other embeddings.

    Args:
        temperature : τ — controls sharpness of the distribution
    """

    def __init__(self, temperature: float = 0.07) -> None:
        super().__init__()
        self.tau = temperature

    def forward(self, z_i: torch.Tensor, z_j: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z_i : (batch_num, out_dim) L2-normalised embeddings for view 1
            z_j : (batch_num, out_dim) L2-normalised embeddings for view 2

        Returns:
            loss : Scalar NT-Xent loss averaged over all 2N embeddings
        """
        batch_num = z_i.size(0)

        # Concatenate both views into a single batch of 2N embeddings
        # (2*batch_num, out_dim)
        z = torch.cat([z_i, z_j], dim=0)

        # Pairwise cosine similarity matrix (dot product on unit sphere)
        # (2*batch_num, 2*batch_num)
        sim = z @ z.T / self.tau

        # Remove self-similarity from the diagonal (a sample cannot be its own positive)
        mask_self = torch.eye(2 * batch_num, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask_self, float("-inf"))

        # Positive pair labels: for z_i[k], positive is z_j[k] at index k+N;
        # for z_j[k], positive is z_i[k] at index k.
        # (2*batch_num,)
        labels = torch.cat([
            torch.arange(batch_num, 2 * batch_num),  # z_i[k] → z_j[k]
            torch.arange(0, batch_num),               # z_j[k] → z_i[k]
        ]).to(z.device)

        # Cross-entropy over the 2N-1 non-self entries
        # (2*batch_num, 2*batch_num) → scalar
        return F.cross_entropy(sim, labels)


class SimCLR(nn.Module):
    """
    SimCLR: ResNet-50 backbone + non-linear projection head.
    The backbone output (before the head) is the transferable representation.
    """

    def __init__(self, base_encoder: str = "resnet50", out_dim: int = 128) -> None:
        super().__init__()

        backbone    = getattr(models, base_encoder)(weights=None)
        in_features = backbone.fc.in_features

        # Remove the classification head — we want backbone features only
        backbone.fc = nn.Identity()
        self.encoder = backbone

        # Non-linear projection head
        # (batch_num, in_features) → (batch_num, out_dim)
        self.proj_head = ProjectionHead(in_features, hidden_dim=512, out_dim=out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, 3, H, W)

        Returns:
            z : (batch_num, out_dim) L2-normalised projection
        """
        # (batch_num, 3, H, W) → (batch_num, in_features)
        h = self.encoder(x)
        # (batch_num, in_features) → (batch_num, out_dim)
        return self.proj_head(h)


# ── Dry-run ────────────────────────────────────────────────────────────────────
simclr_model = SimCLR(base_encoder="resnet18", out_dim=128)
xi_test = torch.zeros(4, 3, 96, 96)
xj_test = torch.zeros(4, 3, 96, 96)
with torch.no_grad():
    zi, zj = simclr_model(xi_test), simclr_model(xj_test)
print(f"Projection z_i : {zi.shape}")  # (4, 128)
print(f"z_i norm (unit sphere): {zi.norm(dim=1)}")  # should be all 1.0

criterion_simclr = NTXentLoss(temperature=0.07)
loss_test = criterion_simclr(zi, zj)
print(f"NT-Xent loss (random init): {loss_test.item():.4f}")

# 2) SimCLR Training Loop

**One epoch of SimCLR** (abbreviated — full SSL typically requires 100–1000 epochs)

In practice, SimCLR is trained with:
- Large batch sizes (4096–8192) — more negatives per step → better representations
- LARS optimiser — large-batch-stable learning rate scaling
- Long schedules (100–200 epochs on ImageNet)

Here we demonstrate 2 epochs on STL-10 to show the mechanics.

In [ ]:
def pretrain_simclr(
    model: nn.Module,
    loader: DataLoader,
    n_epochs: int = 2,
    lr: float = 1e-3,
) -> list:
    """
    Self-supervised pretraining loop for SimCLR.

    Args:
        model    : SimCLR (encoder + projection head)
        loader   : DataLoader returning (x_i, x_j) augmentation pairs
        n_epochs : Number of pretraining epochs (use 100+ for real SSL)
        lr       : Base learning rate

    Returns:
        loss_history : List of mean epoch losses
    """
    model.to(DEVICE)
    criterion = NTXentLoss(temperature=0.07)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0

        for step, (xi, xj) in enumerate(loader):
            xi, xj = xi.to(DEVICE), xj.to(DEVICE)
            optimizer.zero_grad()

            # Forward both views through the same encoder + head
            # (batch_num, 3, H, W) → (batch_num, out_dim)
            zi = model(xi)
            zj = model(xj)

            # NT-Xent operates on the normalised projections
            loss = criterion(zi, zj)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if step % 50 == 0:
                print(f"  Epoch {epoch}/{n_epochs} Step {step}/{len(loader)} loss={loss.item():.4f}")

        scheduler.step()
        mean_loss = running_loss / len(loader)
        history.append(mean_loss)
        print(f"Epoch {epoch}/{n_epochs} | mean_loss={mean_loss:.4f}")

    return history


simclr_model_tr = SimCLR(base_encoder="resnet18", out_dim=128)
print(f"SimCLR parameters: {sum(p.numel() for p in simclr_model_tr.parameters()):,}")
print("Starting 2-epoch SimCLR pretraining on STL-10 unlabelled...")
history_ssl = pretrain_simclr(simclr_model_tr, simclr_loader, n_epochs=2, lr=3e-4)

# Extract the pretrained backbone (discard projection head)
pretrained_backbone = simclr_model_tr.encoder
print("\nPretrained backbone ready for downstream fine-tuning.")

# 3) Linear Evaluation Protocol

After SSL pretraining, we freeze the backbone and train only a linear classifier
on top — this tests the *quality of the learned representations* in isolation.

**Why linear evaluation?**
A linear classifier cannot learn non-linear representations. If it works well,
the backbone has already organised the feature space linearly — evidence of
high-quality, transferable representations.

This is the standard SimCLR evaluation protocol.

In [ ]:
@torch.no_grad()
def extract_features_ssl(backbone, loader, device):
    """Extracts backbone features for all samples in the loader."""
    backbone.eval().to(device)
    feats, labels = [], []
    for X, y in loader:
        X = X.to(device)
        # (batch_num, 3, H, W) → (batch_num, feature_dim)
        feats.append(backbone(X).cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)


# STL-10 labelled split for linear evaluation
eval_transform = transforms.Compose([
    transforms.Resize(96),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
stl_train_lbl = torchvision.datasets.STL10("./data", split="train", download=True, transform=eval_transform)
stl_test_lbl  = torchvision.datasets.STL10("./data", split="test",  download=True, transform=eval_transform)
lbl_tr_loader = DataLoader(stl_train_lbl, batch_size=128, shuffle=False, num_workers=2)
lbl_vl_loader = DataLoader(stl_test_lbl,  batch_size=128, shuffle=False, num_workers=2)

# Extract features using the pretrained backbone
print("Extracting SSL features...")
X_tr_ssl, y_tr_ssl = extract_features_ssl(pretrained_backbone, lbl_tr_loader, DEVICE)
X_te_ssl, y_te_ssl = extract_features_ssl(pretrained_backbone, lbl_vl_loader, DEVICE)

print(f"Train features : {X_tr_ssl.shape}")
print(f"Test  features : {X_te_ssl.shape}")

# Train a linear classifier on frozen features
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

scaler        = StandardScaler()
X_tr_sc       = scaler.fit_transform(X_tr_ssl.numpy())
X_te_sc       = scaler.transform(X_te_ssl.numpy())

lin_clf = LogisticRegression(max_iter=200, C=0.1, solver="lbfgs", multi_class="multinomial")
lin_clf.fit(X_tr_sc, y_tr_ssl.numpy())

acc_ssl = accuracy_score(y_te_ssl.numpy(), lin_clf.predict(X_te_sc))
print(f"\nLinear evaluation accuracy (2-epoch SSL, resnet18): {acc_ssl:.3f}")
print("(Expected: ~35-45% with only 2 epochs; ~80%+ after 200 epochs with large batch)")

# 4) CLIP — Contrastive Language–Image Pretraining

**What CLIP adds** (Radford et al., 2021)

SimCLR uses image–image pairs. CLIP uses image–text pairs from the internet,
aligning visual and language representations in a shared embedding space.

**NT-Xent on image–text pairs**
Given a batch of N (image, text) pairs:
- Positive: the matched (image_i, text_i)
- Negatives: all N-1 other texts for image_i, and all N-1 other images for text_i

The loss is symmetric — computed over rows (image→text) and columns (text→image)
of the similarity matrix.

**Why CLIP representations transfer so broadly**
Text descriptions are rich supervisory signals. The same visual concept
is described in many ways in different captions, forcing the image encoder
to learn *semantic* rather than *stylistic* features.

**Sample I/O**
```
images : (batch_num, 3, 224, 224)
texts  : list of batch_num strings
→ image_features : (batch_num, 512)   — L2-normalised
→ text_features  : (batch_num, 512)   — L2-normalised
→ similarity     : (batch_num, batch_num)
```

In [ ]:
# Install: !pip install openai-clip --quiet

import clip

# Load pretrained CLIP ViT-B/32
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model.eval()

print(f"CLIP image encoder : ViT-B/32")
print(f"Embedding dim      : {clip_model.visual.output_dim}")

# ── Zero-shot classification ───────────────────────────────────────────────────
# CLIP can classify images without any fine-tuning by comparing image embeddings
# to text template embeddings.

stl10_classes = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck"
]

# Build text embeddings for each class
text_prompts = clip.tokenize([f"a photo of a {c}" for c in stl10_classes]).to(DEVICE)
with torch.no_grad():
    # (num_classes, embed_dim)
    text_features = clip_model.encode_text(text_prompts)
    text_features = F.normalize(text_features, p=2, dim=1)

# Load a sample test batch and compute image embeddings
sample_loader = DataLoader(stl_test_lbl, batch_size=64, shuffle=True)
imgs_clip, labels_clip = next(iter(sample_loader))

# CLIP expects images preprocessed with clip_preprocess (224×224, specific normalisation)
# For this demo we use a raw STL10 sample and resize
raw_imgs = [stl_test_lbl[i][0] for i in range(64)]
clip_inputs = torch.stack(raw_imgs).to(DEVICE)
clip_inputs = F.interpolate(clip_inputs, size=(224, 224), mode="bilinear", align_corners=False)

with torch.no_grad():
    # (batch_num, 3, 224, 224) → (batch_num, embed_dim)
    image_features = clip_model.encode_image(clip_inputs)
    image_features = F.normalize(image_features, p=2, dim=1)

# Cosine similarity between each image and each class text prompt
# (batch_num, embed_dim) × (embed_dim, num_classes) → (batch_num, num_classes)
logits_clip = (image_features @ text_features.T) * clip_model.logit_scale.exp()
preds_clip  = logits_clip.argmax(dim=1).cpu()

acc_clip = (preds_clip == labels_clip).float().mean().item()
print(f"\nCLIP zero-shot accuracy on STL-10 sample: {acc_clip:.3f}")
print("(~97% SOTA zero-shot; ~70-80% is typical for ViT-B/32)")

# 5) CLIP Training Loss (from Scratch)

Here we implement the CLIP contrastive objective from scratch.
This is the loss that would be used to *train* CLIP, not just to run inference.
Training CLIP requires 400M+ image-text pairs — here we show the mathematics.

In [ ]:
class CLIPLoss(nn.Module):
    """
    Symmetric contrastive loss for image-text alignment (the CLIP objective).

    Given N matched (image, text) pairs:
    - Image loss: for each image, predict which text matches
    - Text  loss: for each text,  predict which image matches
    - Total loss: mean of both directions

    This symmetric formulation is crucial — it ensures the visual and language
    encoders are trained with equal gradients.

    Args:
        temperature : Learnable or fixed temperature parameter
    """

    def __init__(self, temperature: float = 0.07) -> None:
        super().__init__()
        # In CLIP, temperature is a learnable scalar initialised to 0.07
        self.log_temperature = nn.Parameter(torch.ones([]) * np.log(1 / temperature))

    def forward(
        self,
        image_features: torch.Tensor,
        text_features:  torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            image_features : (batch_num, embed_dim) L2-normalised image embeddings
            text_features  : (batch_num, embed_dim) L2-normalised text embeddings

        Returns:
            loss : Scalar symmetric CLIP loss
        """
        # Learnable temperature (clamped to prevent numerical instability)
        temperature = self.log_temperature.exp().clamp(max=100)

        # Pairwise cosine similarity × temperature
        # (batch_num, embed_dim) × (embed_dim, batch_num) → (batch_num, batch_num)
        logits = (image_features @ text_features.T) * temperature

        # Diagonal entries are the correct pairs — labels are [0, 1, 2, ..., N-1]
        batch_num = image_features.size(0)
        labels    = torch.arange(batch_num, device=image_features.device)

        # Image → Text direction: for each image, which text is the match?
        # (batch_num, batch_num) → scalar
        loss_i2t = F.cross_entropy(logits,   labels)

        # Text → Image direction: for each text, which image is the match?
        # (batch_num, batch_num) → scalar
        loss_t2i = F.cross_entropy(logits.T, labels)

        return (loss_i2t + loss_t2i) / 2.0


# ── Synthetic demonstration ────────────────────────────────────────────────────
batch_num  = 8
embed_dim  = 512

clip_loss_fn = CLIPLoss(temperature=0.07)

# Random normalised features (simulate untrained model)
img_feats  = F.normalize(torch.randn(batch_num, embed_dim), dim=1)
text_feats = F.normalize(torch.randn(batch_num, embed_dim), dim=1)
loss_random = clip_loss_fn(img_feats, text_feats)

# Perfect features: image and text are identical for matched pairs
perfect_feats = F.normalize(torch.randn(batch_num, embed_dim), dim=1)
loss_perfect  = clip_loss_fn(perfect_feats, perfect_feats)

print(f"CLIP loss (random features, untrained)  : {loss_random.item():.4f}")
print(f"CLIP loss (perfect alignment)            : {loss_perfect.item():.4f}")
print(f"Random baseline loss ≈ log(N)            : {np.log(batch_num):.4f}")

# 6) DINOv2 — Feature Extraction

**DINOv2** (Oquab et al., 2023) achieves the best general-purpose visual
features by training a ViT with a self-distillation objective on a curated
142M image dataset.

**How DINOv2 differs from SimCLR**
- No explicit negative pairs — avoids batch-size sensitivity
- Student–teacher distillation: teacher is an exponential moving average (EMA)
  of the student weights (momentum encoder)
- Patch-level features (not just CLS) — enables dense prediction
- Trained with multiple crop scales simultaneously

**Why DINOv2 features are superior**
DINOv2's patch-level features form semantically meaningful segments without
any segmentation supervision — a property that emerges purely from SSL.

In [ ]:
# DINOv2 is available through the official Facebook Research hub
# !pip install timm --quiet

import torch.hub

# Load pretrained DINOv2-ViT-B/14 (the standard-size variant)
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", pretrained=True)
dinov2.eval().to(DEVICE)

print(f"DINOv2-ViT-B/14 parameters: {sum(p.numel() for p in dinov2.parameters()):,}")
print(f"Expected output dim: 768  (ViT-B hidden dim)")

# ── Extract features from STL-10 test samples ─────────────────────────────────
dino_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
stl_dino_ds = torchvision.datasets.STL10("./data", split="test", download=True, transform=dino_transform)
dino_loader  = DataLoader(stl_dino_ds, batch_size=64, shuffle=False, num_workers=2)

dino_feats, dino_labels = [], []
with torch.no_grad():
    for imgs, lbls in dino_loader:
        imgs = imgs.to(DEVICE)
        # (batch_num, 3, 224, 224) → (batch_num, 768)  — CLS token features
        out = dinov2(imgs)
        dino_feats.append(out.cpu())
        dino_labels.append(lbls)
        if len(dino_feats) * 64 >= 512:  # limit for demo
            break

dino_feats  = torch.cat(dino_feats)[:512]
dino_labels = torch.cat(dino_labels)[:512]
print(f"\nExtracted DINOv2 features: {dino_feats.shape}")

# KNN classification on DINOv2 features
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

sc        = StandardScaler()
X_dino_sc = sc.fit_transform(dino_feats.numpy())
knn       = KNeighborsClassifier(n_neighbors=5, metric="cosine")
knn.fit(X_dino_sc[:400], dino_labels[:400].numpy())
acc_dino  = accuracy_score(dino_labels[400:].numpy(), knn.predict(X_dino_sc[400:]))
print(f"5-NN accuracy on DINOv2 features (512 samples): {acc_dino:.3f}")
print("(DINOv2 achieves ~86% on full STL-10 with kNN — no fine-tuning)")

# 7) Visualising DINOv2 Patch Features

One of the most striking properties of DINOv2 is that its patch-level features
form semantically coherent regions without segmentation supervision.

By computing the PCA of patch features and colouring patches by their PCA
component, we see that the model has implicitly segmented the main subject
from the background.\


In [ ]:
from sklearn.decomposition import PCA

# Get patch-level features (not CLS) using forward_features
img_vis, _ = stl_dino_ds[0]
img_vis     = img_vis.unsqueeze(0).to(DEVICE)  # (1, 3, 224, 224)

with torch.no_grad():
    # DINOv2 returns a dict with 'x_norm_patchtokens' when called with forward_features
    feat_dict = dinov2.forward_features(img_vis)
    # patch tokens: (1, num_patches, hidden_dim)  — excludes CLS
    patch_feats = feat_dict["x_norm_patchtokens"].squeeze(0).cpu().numpy()  # (num_patches, 768)

# For ViT-B/14 on 224×224: patch_size=14 → 16×16=256 patches
num_patches = patch_feats.shape[0]
grid_size   = int(num_patches ** 0.5)
print(f"Patch features : {patch_feats.shape}")  # (256, 768)
print(f"Grid size      : {grid_size}×{grid_size}")

# PCA: project 768-dim features to 3 components (RGB channels for visualisation)
pca = PCA(n_components=3)
pca_feats = pca.fit_transform(patch_feats)  # (num_patches, 3)

# Normalise each component to [0, 1] for display
for c in range(3):
    pca_feats[:, c] = (pca_feats[:, c] - pca_feats[:, c].min()) / (pca_feats[:, c].ptp() + 1e-8)

# Reshape to grid
pca_img = pca_feats.reshape(grid_size, grid_size, 3)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_vis.squeeze().cpu().permute(1,2,0).numpy() * 0.2 + 0.5)  # rough denorm
axes[0].set_title("Input image"); axes[0].axis("off")
axes[1].imshow(pca_img)
axes[1].set_title("DINOv2 patch features (PCA 3-component RGB)"); axes[1].axis("off")
plt.suptitle("Emergent segmentation from SSL pretraining — no labels used")
plt.tight_layout(); plt.show()

# 8) CLIP vs SigLIP — Why the Field Moved from InfoNCE to Sigmoid

Both CLIP and SigLIP train two encoders (image + text) so that matching
pairs are close in embedding space and non-matching pairs are far apart.
They differ in exactly one place: **the loss function**.

That one difference has significant engineering consequences:

| | CLIP (Radford 2021) | SigLIP (Zhai 2023) |
|---|---|---|
| Loss | InfoNCE (softmax over batch) | Sigmoid per pair (binary CE) |
| Batch size needed | 32 768 in original paper | Works at 1 024–4 096 |
| Distributed training | Needs all-gather across GPUs | Each GPU is independent |
| Hard negatives | Implicit via large batch | Explicit via sampling |
| Adopted by | Original CLIP, early LLaVA | Qwen2.5-VL, InternVL3, Gemma 3, PaLI |

**This section builds both from scratch** — same dual-encoder, same data,
only the loss changes — so the comparison is controlled and interpretable.

## 8.1 Dual-Encoder Architecture

Both CLIP and SigLIP use a **shared dual-encoder** structure:

```
Image x_i ──► ImageEncoder ──► image_emb_i  (embed_dim,) L2-normalised
Text  t_i ──► TextEncoder  ──► text_emb_i   (embed_dim,) L2-normalised

For a batch of N pairs:
  image_embs : (N, embed_dim)
  text_embs  : (N, embed_dim)
  similarity : image_embs @ text_embs.T → (N, N)
```

**L2 normalisation** is essential — it constrains embeddings to the unit
hypersphere, making cosine similarity equal to the dot product and keeping
loss gradients well-scaled regardless of embedding magnitude.

In [ ]:
import random
import math

# ── Vocabulary for character-level text tokenisation ──────────────────────────
_CHARS = list("abcdefghijklmnopqrstuvwxyz0123456789 .,'-")
PAD_IDX, CLS_IDX = len(_CHARS), len(_CHARS) + 1
VOCAB_SIZE        = len(_CHARS) + 2   # characters + PAD + CLS
MAX_TEXT_LEN      = 40

_char2idx = {c: i for i, c in enumerate(_CHARS)}

def tokenize(text: str, max_len: int = MAX_TEXT_LEN) -> torch.Tensor:
    """
    Character-level tokenisation.

    Prepends a [CLS] token (used for sequence-level representation),
    truncates to max_len, and pads to a fixed length.

    Args:
        text    : Input string (lowercased internally)
        max_len : Fixed output length

    Returns:
        tokens : (max_len,) long tensor
    """
    text   = text.lower().strip()[:max_len - 1]
    ids    = [CLS_IDX] + [_char2idx.get(c, PAD_IDX) for c in text]
    ids   += [PAD_IDX] * (max_len - len(ids))
    return torch.tensor(ids[:max_len], dtype=torch.long)


class MiniImageEncoder(nn.Module):
    """
    Lightweight image encoder: ResNet-18 backbone → linear projection.

    Trained from scratch here (no pretrained weights) so that both CLIP
    and SigLIP start from the same random initialisation.
    """

    def __init__(self, embed_dim: int = 128) -> None:
        super().__init__()
        backbone     = models.resnet18(weights=None)
        backbone.fc  = nn.Identity()        # remove the 1000-class head
        self.backbone = backbone            # outputs (batch_num, 512)
        # Project backbone features to shared embedding space
        # (batch_num, 512) → (batch_num, embed_dim)
        self.proj = nn.Linear(512, embed_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, 3, H, W)

        Returns:
            emb : (batch_num, embed_dim)  L2-normalised
        """
        # (batch_num, 3, H, W) → (batch_num, 512)
        h = self.backbone(x)
        # (batch_num, 512) → (batch_num, embed_dim)
        z = self.proj(h)
        # Project onto unit hypersphere — required for cosine similarity loss
        # (batch_num, embed_dim) → (batch_num, embed_dim)
        return F.normalize(z, p=2, dim=-1)


class MiniTextEncoder(nn.Module):
    """
    Lightweight text encoder: character embedding → Transformer → CLS → projection.

    Deliberately small (2 layers, 128 hidden) so it trains in minutes on CPU.
    Real models (CLIP, SigLIP) use a 12–24 layer text Transformer.
    """

    def __init__(
        self,
        vocab_size: int  = VOCAB_SIZE,
        model_dim: int   = 128,
        embed_dim: int   = 128,
        num_heads: int   = 4,
        num_layers: int  = 2,
        max_len: int     = MAX_TEXT_LEN,
    ) -> None:
        super().__init__()

        # Character-level token embedding + learned positional embedding
        # (batch_num, seq_len) → (batch_num, seq_len, model_dim)
        self.tok_embed = nn.Embedding(vocab_size, model_dim, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(max_len, model_dim)

        # Small Transformer encoder — batch_first so input is (B, T, C)
        enc_layer      = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Final projection to shared embedding space
        # (batch_num, model_dim) → (batch_num, embed_dim)
        self.proj = nn.Linear(model_dim, embed_dim, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """
        Args:
            tokens : (batch_num, seq_len) long tensor from tokenize()

        Returns:
            emb : (batch_num, embed_dim)  L2-normalised
        """
        batch_num, seq_len = tokens.shape
        positions = torch.arange(seq_len, device=tokens.device)

        # Token + positional embeddings
        # (batch_num, seq_len, model_dim)
        x = self.tok_embed(tokens) + self.pos_embed(positions)

        # Padding mask: True = position should be ignored
        # (batch_num, seq_len) — True where PAD_IDX
        pad_mask = tokens == PAD_IDX

        # (batch_num, seq_len, model_dim)
        x = self.transformer(x, src_key_padding_mask=pad_mask)

        # CLS token (position 0) aggregates the full sequence
        # (batch_num, model_dim)
        cls_out = x[:, 0]

        # Project and L2-normalise
        # (batch_num, embed_dim)
        return F.normalize(self.proj(cls_out), p=2, dim=-1)


# ── Shape verification ─────────────────────────────────────────────────────────
img_enc  = MiniImageEncoder(embed_dim=128)
txt_enc  = MiniTextEncoder(embed_dim=128)

imgs_test  = torch.zeros(4, 3, 32, 32)
texts_test = torch.stack([tokenize("a photo of a cat") for _ in range(4)])

with torch.no_grad():
    img_emb = img_enc(imgs_test)
    txt_emb = txt_enc(texts_test)

print(f"Image embeddings : {img_emb.shape}  norms={img_emb.norm(dim=1).tolist()}")
print(f"Text  embeddings : {txt_emb.shape}  norms={txt_emb.norm(dim=1).tolist()}")

img_params = sum(p.numel() for p in img_enc.parameters())
txt_params = sum(p.numel() for p in txt_enc.parameters())
print(f"\nImageEncoder params : {img_params:,}")
print(f"TextEncoder  params : {txt_params:,}")

## 8.2 CLIP Loss — InfoNCE (Softmax over Full Batch)

**The InfoNCE objective**

For a batch of $N$ image-text pairs, the loss for image $i$ is:

$$L_i^{i \to t} = -\log \frac{\exp(\text{sim}(z_i^I, z_i^T)/\tau)}{\sum_{j=1}^{N} \exp(\text{sim}(z_i^I, z_j^T)/\tau)}$$

and symmetrically for text. The full loss averages both directions.

**Why this needs large batches**

The denominator is a normalisation over *all* $N$ texts in the batch.
When $N$ is small (e.g., 64), most negatives are easy (very different images),
so the gradient signal concentrates on the one positive and barely touches
the easy negatives.

Original CLIP used $N = 32{,}768$ — 512 GPUs × 64 images each.
At that scale, every batch contains genuinely hard negatives by chance.

**Learnable temperature $\tau$**

In the original CLIP paper, $\tau$ starts at $0.07$ and is learned.
Too small: softmax becomes one-hot → only the hardest negative gets gradient.
Too large: softmax is uniform → no signal from easy negatives either.
The learned $\tau$ finds the right balance automatically.

In [ ]:
class CLIPLoss(nn.Module):
    """
    InfoNCE contrastive loss for image-text alignment.

    Treats the N-1 non-matching pairs within the batch as negatives.
    The loss equals log(N) for a randomly initialised model (uniform softmax),
    which is a useful calibration: lower is better, 0 is perfect.

    Args:
        init_temperature : Starting value of the learnable temperature τ
    """

    def __init__(self, init_temperature: float = 0.07) -> None:
        super().__init__()
        # Store log(τ) as the parameter — ensures τ stays positive
        self.log_tau = nn.Parameter(torch.tensor(math.log(init_temperature)))

    @property
    def temperature(self) -> torch.Tensor:
        """Clamp to [0.01, 100] for numerical stability."""
        return self.log_tau.exp().clamp(min=0.01, max=100.0)

    def forward(
        self, image_emb: torch.Tensor, text_emb: torch.Tensor
    ) -> tuple:
        """
        Args:
            image_emb : (batch_num, embed_dim)  L2-normalised image embeddings
            text_emb  : (batch_num, embed_dim)  L2-normalised text embeddings

        Returns:
            loss : Scalar mean InfoNCE loss (symmetric)
            logits : (batch_num, batch_num) similarity matrix (for inspection)
        """
        batch_num = image_emb.size(0)

        # Scaled cosine similarity — on unit sphere this equals dot product / τ
        # (batch_num, embed_dim) × (embed_dim, batch_num) → (batch_num, batch_num)
        logits = (image_emb @ text_emb.T) / self.temperature

        # Ground truth: row i should be most similar to column i
        # (batch_num,)
        labels = torch.arange(batch_num, device=image_emb.device)

        # Image → text: for each image, which text matches?
        # (batch_num, batch_num) → scalar
        loss_i2t = F.cross_entropy(logits,   labels)
        # Text → image: for each text, which image matches?
        # (batch_num, batch_num) → scalar
        loss_t2i = F.cross_entropy(logits.T, labels)

        return (loss_i2t + loss_t2i) / 2.0, logits


# Calibration: untrained model should give loss ≈ log(N)
clip_loss_fn = CLIPLoss(init_temperature=0.07)
img_rand = F.normalize(torch.randn(64, 128), dim=-1)
txt_rand = F.normalize(torch.randn(64, 128), dim=-1)

with torch.no_grad():
    loss_rand, _ = clip_loss_fn(img_rand, txt_rand)

print(f"CLIP loss (random init, N=64) : {loss_rand.item():.4f}")
print(f"Expected ≈ log(N) = log(64)  : {math.log(64):.4f}  ✓" if abs(loss_rand.item() - math.log(64)) < 0.5 else "")
print(f"Learnable temperature τ       : {clip_loss_fn.temperature.item():.4f}")

## 8.3 SigLIP Loss — Sigmoid Binary Cross-Entropy per Pair

**The SigLIP objective** (Zhai et al., 2023)

Instead of normalising over the full batch, SigLIP treats each of the $N^2$
pairs as an **independent binary classification** problem:
- Diagonal entries (matching pairs): label = $+1$ → push $\sigma(s_{ij}) \to 1$
- Off-diagonal entries (non-matching): label = $-1$ → push $\sigma(s_{ij}) \to 0$

$$L = -\frac{1}{N^2} \sum_{i=1}^{N} \sum_{j=1}^{N} \log \sigma\!\left(y_{ij} \cdot (s_{ij} \cdot t + b)\right)$$

where $y_{ij} = +1$ if $i=j$ else $-1$, $t$ is a learnable temperature, and $b$ is a learnable bias.

**Why sigmoid removes the large-batch requirement**

In InfoNCE, the gradient for pair $(i, j)$ depends on the softmax distribution
over all $N$ items — every other item in the batch affects the gradient.
With sigmoid, pair $(i, j)$'s gradient is **independent** of all other pairs.
Each GPU can compute its local gradient without communicating the full logit matrix.

**The bias term $b$**

SigLIP adds a learnable scalar bias to all logits.
Initialised to a large negative value (e.g., $-10$) so that all pairs start
near $\sigma(0) = 0.5$ rather than $\sigma(\text{large}) \approx 1$.
This prevents the model from getting stuck predicting all pairs as matching.

In [ ]:
class SigLIPLoss(nn.Module):
    """
    Sigmoid binary cross-entropy loss for image-text alignment.

    Each of the N² pairs is an independent binary classification:
    matching pair → label +1, non-matching → label -1.

    Advantages over InfoNCE:
    - No all-gather needed across GPUs (each pair is independent)
    - Works well at smaller batch sizes
    - Gradient variance is lower (N² pairs vs N effective pairs in InfoNCE)

    Args:
        init_temperature : Starting scale for logits (not 1/τ — here it's τ directly)
        init_bias        : Additive bias initialised negative so sigmoid ≈ 0.5 at start
    """

    def __init__(self, init_temperature: float = 10.0, init_bias: float = -10.0) -> None:
        super().__init__()
        # Temperature is a scale factor (multiply, not divide) — Zhai et al. convention
        self.log_temp = nn.Parameter(torch.tensor(math.log(init_temperature)))
        # Global additive bias — shifts the operating point of all sigmoid activations
        self.bias     = nn.Parameter(torch.tensor(init_bias))

    @property
    def temperature(self) -> torch.Tensor:
        return self.log_temp.exp().clamp(min=0.1, max=1000.0)

    def forward(
        self, image_emb: torch.Tensor, text_emb: torch.Tensor
    ) -> tuple:
        """
        Args:
            image_emb : (batch_num, embed_dim)  L2-normalised image embeddings
            text_emb  : (batch_num, embed_dim)  L2-normalised text embeddings

        Returns:
            loss   : Scalar mean SigLIP loss
            logits : (batch_num, batch_num) scaled similarity matrix
        """
        # Scaled pairwise cosine similarity + global bias
        # (batch_num, embed_dim) × (embed_dim, batch_num) → (batch_num, batch_num)
        logits = image_emb @ text_emb.T * self.temperature + self.bias

        # Binary labels: +1 on diagonal (matching), -1 everywhere else
        # (batch_num, batch_num)
        labels = 2.0 * torch.eye(logits.size(0), device=logits.device) - 1.0

        # Per-pair sigmoid binary cross-entropy: -log σ(y · s)
        # F.logsigmoid(x) = log(σ(x)) — numerically stable
        # (batch_num, batch_num) → scalar
        loss = -F.logsigmoid(labels * logits).mean()

        return loss, logits


# Calibration check
siglip_loss_fn = SigLIPLoss(init_temperature=10.0, init_bias=-10.0)
with torch.no_grad():
    loss_sig, _ = siglip_loss_fn(img_rand, txt_rand)

print(f"SigLIP loss (random init, N=64) : {loss_sig.item():.4f}")
print(f"Expected ≈ log(2) = {math.log(2):.4f} (balanced binary CE at initialisation)")
print(f"Learnable temperature            : {siglip_loss_fn.temperature.item():.2f}")
print(f"Learnable bias                   : {siglip_loss_fn.bias.item():.2f}")

## 8.4 Dataset — CIFAR-10 with Caption Templates

To keep this notebook self-contained (no large downloads), we pair CIFAR-10
images with templated captions: `"a photo of a {class_name}"`.

This is not as rich as real image-text pairs, but it exercises the exact
same training mechanics — and lets us run controlled CLIP vs SigLIP comparisons
with identical data.

**Augmentation for image-text contrastive training**
Unlike SimCLR (which pairs two augmentations of the *same* image),
CLIP/SigLIP pair an image with its matched caption.
Augmentation still helps but the positive signal is the semantic match.

In [ ]:
CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

CAPTION_TEMPLATES = [
    "a photo of a {}",
    "an image of a {}",
    "a clear picture of a {}",
    "a {} in the photo",
]


class CIFARCaptionDataset(Dataset):
    """
    Wraps CIFAR-10 and returns (image, token_ids, class_idx) triples.

    Each image is paired with a randomly sampled caption template for its class.
    This gives N distinct captions per class across the epoch (data augmentation
    in the text modality).
    """

    def __init__(self, split: str = "train") -> None:
        tf = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.base = torchvision.datasets.CIFAR10(
            "./data", train=(split == "train"), download=True, transform=tf
        )

    def __len__(self): return len(self.base)

    def __getitem__(self, idx) -> tuple:
        img, label = self.base[idx]
        caption    = random.choice(CAPTION_TEMPLATES).format(CIFAR_CLASSES[label])
        # (1, MAX_TEXT_LEN)
        tokens     = tokenize(caption)
        return img, tokens, label


# Pre-tokenise all 10 class prompts for zero-shot evaluation (Section 8.6)
def build_class_text_embeddings(
    text_encoder: nn.Module,
    classes: list,
    template: str = "a photo of a {}",
    device: torch.device = DEVICE,
) -> torch.Tensor:
    """
    Encodes one caption per class and returns normalised text embeddings.

    Returns:
        class_embs : (num_classes, embed_dim)  L2-normalised
    """
    text_encoder.eval().to(device)
    tokens = torch.stack([tokenize(template.format(c)) for c in classes]).to(device)
    with torch.no_grad():
        # (num_classes, MAX_TEXT_LEN) → (num_classes, embed_dim)
        embs = text_encoder(tokens)
    return embs


# Build datasets
train_ds   = CIFARCaptionDataset("train")
val_ds     = CIFARCaptionDataset("test")
train_dl   = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=2, drop_last=True)
val_dl     = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2, drop_last=True)

img_s, tok_s, lbl_s = train_ds[0]
print(f"Image shape  : {img_s.shape}")
print(f"Token shape  : {tok_s.shape}")
print(f"Caption      : {CAPTION_TEMPLATES[0].format(CIFAR_CLASSES[lbl_s])}")
print(f"Tokens[:10]  : {tok_s[:10].tolist()}")

## 8.5 Training Comparison — CLIP vs SigLIP

We train **two model pairs** (each has a fresh `MiniImageEncoder` + `MiniTextEncoder`)
on the exact same data with the exact same optimiser settings.
The only difference is the loss function.

**What to watch**
- `loss`: should decrease for both
- `zero_shot_acc`: retrieval accuracy using text embeddings for all 10 CIFAR classes
- **Convergence speed**: SigLIP should reach comparable accuracy with fewer epochs
  because each pair provides an independent gradient signal — no gradient averaging
  over N negatives as in InfoNCE

In [ ]:
def make_encoders(embed_dim: int = 128) -> tuple:
    """Returns a fresh (image_encoder, text_encoder) pair."""
    return MiniImageEncoder(embed_dim).to(DEVICE), MiniTextEncoder(embed_dim=embed_dim).to(DEVICE)


@torch.no_grad()
def zero_shot_accuracy(
    img_enc: nn.Module,
    txt_enc: nn.Module,
    loader: DataLoader,
    device: torch.device = DEVICE,
) -> float:
    """
    Computes zero-shot classification accuracy.

    For each image, computes cosine similarity against text embeddings for all
    10 CIFAR-10 class prompts and predicts the most similar class.
    """
    img_enc.eval(); txt_enc.eval()

    # Build 10 class text embeddings from the default template
    # (10, embed_dim)
    class_embs = build_class_text_embeddings(txt_enc, CIFAR_CLASSES, device=device)

    correct = 0
    for imgs, _, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        # (batch_num, embed_dim)
        img_embs = img_enc(imgs)
        # Cosine similarity to each class text embedding
        # (batch_num, embed_dim) × (embed_dim, 10) → (batch_num, 10)
        sims     = img_embs @ class_embs.T
        preds    = sims.argmax(dim=1)
        correct += (preds == labels).sum().item()

    return correct / len(loader.dataset)


def train_one_run(
    img_enc: nn.Module,
    txt_enc: nn.Module,
    loss_fn: nn.Module,
    train_dl: DataLoader,
    val_dl: DataLoader,
    n_epochs: int = 5,
    lr: float = 3e-4,
    label: str = "model",
) -> dict:
    """
    Generic training loop for both CLIP and SigLIP.
    Returns history dict.
    """
    params = (list(img_enc.parameters())
            + list(txt_enc.parameters())
            + list(loss_fn.parameters()))
    optimizer = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    history   = {"loss": [], "zero_shot_acc": [], "temperature": []}

    for epoch in range(1, n_epochs + 1):
        img_enc.train(); txt_enc.train(); loss_fn.train()
        running_loss = 0.0

        for imgs, tokens, _ in train_dl:
            imgs, tokens = imgs.to(DEVICE), tokens.to(DEVICE)
            optimizer.zero_grad()

            # (batch_num, 3, 32, 32) → (batch_num, embed_dim)
            img_emb = img_enc(imgs)
            # (batch_num, MAX_TEXT_LEN) → (batch_num, embed_dim)
            txt_emb = txt_enc(tokens)

            loss, _ = loss_fn(img_emb, txt_emb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()

        zs_acc = zero_shot_accuracy(img_enc, txt_enc, val_dl)
        tl     = running_loss / len(train_dl.dataset)
        tau    = loss_fn.temperature.item()

        history["loss"].append(tl)
        history["zero_shot_acc"].append(zs_acc)
        history["temperature"].append(tau)
        print(f"[{label}] Epoch {epoch}/{n_epochs} | loss={tl:.4f} | "
              f"zero_shot_acc={zs_acc:.3f} | τ={tau:.3f}")

    return history


# ── Train CLIP ─────────────────────────────────────────────────────────────────
print("=" * 55)
print("Training CLIP (InfoNCE)")
print("=" * 55)
torch.manual_seed(0)
clip_img, clip_txt = make_encoders(embed_dim=128)
clip_loss   = CLIPLoss(init_temperature=0.07)
hist_clip   = train_one_run(clip_img, clip_txt, clip_loss,
                             train_dl, val_dl, n_epochs=5, label="CLIP")

# ── Train SigLIP ───────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("Training SigLIP (Sigmoid BCE)")
print("=" * 55)
torch.manual_seed(0)
sig_img, sig_txt = make_encoders(embed_dim=128)
sig_loss    = SigLIPLoss(init_temperature=10.0, init_bias=-10.0)
hist_siglip = train_one_run(sig_img, sig_txt, sig_loss,
                             train_dl, val_dl, n_epochs=5, label="SigLIP")

## 8.6 Batch Size Analysis — Why SigLIP Wins at Small Batches

**The gradient signal perspective**

In InfoNCE with batch size $N$:
- Each anchor sees $N-1$ negatives
- The gradient for the positive pair is diluted by $N-1$ negatives
- Effective gradient per pair $\propto 1/(N-1)$ — decreases as batch shrinks

In SigLIP sigmoid BCE with batch size $N$:
- There are $N^2$ binary pairs, of which $N$ are positive and $N(N-1)$ are negative
- Each pair contributes an independent gradient — **no normalisation across batch**
- Total gradient signal $\propto N^2$ (grows quadratically with batch size!)

This means SigLIP gets more gradient signal per GPU even at the same batch size,
and its gradient does not degrade nearly as badly at small $N$.

In [ ]:
# Analytical demonstration: gradient signal as a function of batch size

def clip_positive_gradient_magnitude(N: int, tau: float = 0.07) -> float:
    """
    Expected magnitude of the gradient for a positive pair in CLIP InfoNCE.

    For a random (untrained) model with uniform similarity,
    the softmax probability of the positive = 1/N.
    Gradient of CE loss at that point ≈ (1 - 1/N).
    """
    p_pos = 1.0 / N         # softmax prob at positive under random model
    return 1.0 - p_pos      # CE gradient magnitude for the positive


def siglip_positive_gradient_magnitude(N: int, t: float = 10.0, b: float = -10.0) -> float:
    """
    Expected gradient magnitude for a positive pair in SigLIP.

    Under random model with unit-sphere embeddings:
    mean cosine similarity ≈ 0, so logit ≈ 0 * t + b = b.
    Sigmoid gradient at x: σ(x)(1-σ(x)).
    For positive pairs (label +1): gradient = 1 - σ(b).
    """
    import math
    sig_b = 1 / (1 + math.exp(-b))  # σ(b)
    return 1.0 - sig_b               # gradient magnitude for positive pair


batch_sizes = [8, 16, 32, 64, 128, 256, 512, 1024]
clip_grads   = [clip_positive_gradient_magnitude(N)   for N in batch_sizes]
siglip_grads = [siglip_positive_gradient_magnitude(N) for N in batch_sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(batch_sizes, clip_grads,   "o-", label="CLIP (InfoNCE)",     color="steelblue")
axes[0].plot(batch_sizes, siglip_grads, "s-", label="SigLIP (Sigmoid)",   color="coral")
axes[0].set_title("Positive-pair gradient magnitude vs batch size")
axes[0].set_xlabel("Batch size N")
axes[0].set_ylabel("Gradient magnitude")
axes[0].legend()
axes[0].set_xscale("log")

# Training curves comparison
epochs = list(range(1, 6))
axes[1].plot(epochs, hist_clip["zero_shot_acc"],   "o-", label="CLIP",   color="steelblue")
axes[1].plot(epochs, hist_siglip["zero_shot_acc"], "s-", label="SigLIP", color="coral")
axes[1].set_title("Zero-shot accuracy (N=256, same data)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout(); plt.show()

# Learned temperature comparison
print(f"CLIP   final τ : {hist_clip['temperature'][-1]:.4f}")
print(f"SigLIP final τ : {hist_siglip['temperature'][-1]:.4f}")
print()
print("Note: SigLIP temperature is a *multiplier* (≈ 10–100),")
print("      CLIP   temperature is a *divisor*  (≈ 0.01–0.1).")

## 8.7 Zero-Shot Classification — CLIP vs SigLIP

Both models perform zero-shot classification by comparing image embeddings
to text embeddings of class name prompts — no fine-tuning required.

The prompt templates matter: `"a photo of a dog"` works better than `"dog"` alone.
In practice, ensembling 80 different templates (the CLIP paper approach) gives
a consistent ~3–5% accuracy gain over any single template.

In [ ]:
def zero_shot_classify(
    img_enc: nn.Module,
    txt_enc: nn.Module,
    dataset: Dataset,
    templates: list,
    classes: list,
    n_samples: int = 500,
    device: torch.device = DEVICE,
) -> dict:
    """
    Runs zero-shot classification and returns per-class and overall accuracy.

    Args:
        templates  : List of text templates to ensemble over
        classes    : List of class name strings
        n_samples  : Number of validation samples to evaluate on

    Returns:
        results : dict with 'overall_acc' and per-class breakdown
    """
    img_enc.eval(); txt_enc.eval()

    # Encode all templates × classes and average to get one embedding per class
    all_class_embs = []
    for cls_name in classes:
        toks  = torch.stack([tokenize(t.format(cls_name)) for t in templates]).to(device)
        with torch.no_grad():
            # (num_templates, embed_dim)
            embs = txt_enc(toks)
        # Average-pool over templates → single class embedding → re-normalise
        # (embed_dim,)
        cls_emb = F.normalize(embs.mean(dim=0, keepdim=True), dim=-1)
        all_class_embs.append(cls_emb)

    # (num_classes, embed_dim)
    class_matrix = torch.cat(all_class_embs, dim=0)

    # Evaluate on n_samples
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    all_preds, all_labels = [], []
    total_seen = 0

    for imgs, _, labels in loader:
        imgs = imgs.to(device)
        with torch.no_grad():
            img_embs = img_enc(imgs)  # (batch, embed_dim)
        # (batch, num_classes)
        sims  = img_embs @ class_matrix.T
        preds = sims.argmax(dim=1).cpu()
        all_preds.append(preds); all_labels.append(labels)
        total_seen += len(imgs)
        if total_seen >= n_samples:
            break

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    overall = (all_preds == all_labels).float().mean().item()
    per_class = {
        classes[c]: ((all_preds == c) & (all_labels == c)).sum().item() /
                    max((all_labels == c).sum().item(), 1)
        for c in range(len(classes))
    }
    return {"overall_acc": overall, "per_class": per_class}


# Multiple prompt templates to ensemble
EVAL_TEMPLATES = [
    "a photo of a {}",
    "an image of a {}",
    "a clear photo of a {}",
    "a {} in the picture",
]

print("=== Zero-shot classification (ensembled over 4 templates) ===")
for label, img_e, txt_e in [("CLIP", clip_img, clip_txt), ("SigLIP", sig_img, sig_txt)]:
    res = zero_shot_classify(img_e, txt_e, val_ds, EVAL_TEMPLATES, CIFAR_CLASSES)
    print(f"\n{label}  overall accuracy: {res['overall_acc']:.3f}")
    for cls, acc in res["per_class"].items():
        bar = "█" * int(acc * 20)
        print(f"  {cls:<12} {acc:.2f}  {bar}")

## 8.8 Similarity Matrix Visualisation

The $N \times N$ similarity matrix is the core diagnostic for both CLIP and SigLIP:
- Diagonal should be bright (matching pairs have high similarity)
- Off-diagonal should be dark (non-matching pairs have low similarity)
- Rows/columns within the same class may be partially bright — the model
  correctly knows "dog photo ↔ dog caption" but also "dog photo ≈ dog photo"

In [ ]:
def plot_similarity_matrix(
    img_enc: nn.Module,
    txt_enc: nn.Module,
    dataset: Dataset,
    n: int = 32,
    title: str = "",
    device: torch.device = DEVICE,
) -> None:
    """Plots the N×N cosine similarity matrix for a batch of N pairs."""
    img_enc.eval(); txt_enc.eval()

    loader = DataLoader(dataset, batch_size=n, shuffle=True)
    imgs, tokens, labels = next(iter(loader))
    imgs, tokens = imgs.to(device), tokens.to(device)

    with torch.no_grad():
        img_embs = img_enc(imgs)    # (n, embed_dim)
        txt_embs = txt_enc(tokens)  # (n, embed_dim)
        # (n, n)
        sim_mat = (img_embs @ txt_embs.T).cpu().numpy()

    class_tick_labels = [CIFAR_CLASSES[l] for l in labels.tolist()]

    plt.figure(figsize=(8, 6))
    plt.imshow(sim_mat, cmap="RdBu_r", vmin=-1, vmax=1)
    plt.colorbar(label="cosine similarity")
    plt.xticks(range(n), class_tick_labels, rotation=90, fontsize=5)
    plt.yticks(range(n), class_tick_labels, fontsize=5)
    plt.xlabel("Text embedding (caption)"); plt.ylabel("Image embedding")
    plt.title(f"{title} — N={n} similarity matrix")
    plt.tight_layout(); plt.show()


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, img_e, txt_e) in zip(
    axes.flatten(), [("CLIP", clip_img, clip_txt), ("SigLIP", sig_img, sig_txt)]
):
    plt.sca(ax)
    plot_similarity_matrix(img_e, txt_e, val_ds, n=32, title=label)
plt.suptitle("Similarity matrices: diagonal = correct pairs"); plt.tight_layout(); plt.show()

## 8.9 Pretrained SigLIP via Hugging Face

Google's `google/siglip-base-patch16-224` provides a production-quality SigLIP model.

**Architecture**
- Image encoder: ViT-B/16 (86M parameters)
- Text encoder: 12-layer Transformer with SentencePiece tokenisation
- Embedding dim: 768
- Pretrained on 2B image-text pairs from the web (LAION-style, curated)

**Zero-shot evaluation**
SigLIP's pretrained checkpoint achieves ~76–80% zero-shot accuracy on ImageNet
— higher than CLIP ViT-B/32 (~63%) because:
1. The SigLIP loss admits smaller but more diverse batches → more varied training signal
2. The bias term provides better calibration
3. Training improvements made after the 2021 CLIP paper

In [ ]:
from transformers import AutoProcessor, SiglipModel

# Load pretrained SigLIP (google/siglip-base-patch16-224)
siglip_pretrained = SiglipModel.from_pretrained("google/siglip-base-patch16-224").eval().to(DEVICE)
siglip_processor  = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")

total_params = sum(p.numel() for p in siglip_pretrained.parameters())
print(f"Pretrained SigLIP parameters : {total_params:,}")
print(f"Image encoder                : {sum(p.numel() for p in siglip_pretrained.vision_model.parameters()):,}")
print(f"Text  encoder                : {sum(p.numel() for p in siglip_pretrained.text_model.parameters()):,}")

# ── Zero-shot on a CIFAR-10 sample ────────────────────────────────────────────
# Build class text prompts
prompts = [f"a photo of a {c}" for c in CIFAR_CLASSES]

sample_ds     = torchvision.datasets.CIFAR10("./data", train=False, download=True,
                    transform=transforms.Compose([transforms.Resize(224), transforms.ToTensor()]))
sample_loader = DataLoader(sample_ds, batch_size=64, shuffle=False)

# Encode text prompts
text_inputs = siglip_processor(text=prompts, return_tensors="pt", padding="max_length").to(DEVICE)
with torch.no_grad():
    # (num_classes, embed_dim)
    text_embs = siglip_pretrained.get_text_features(**text_inputs)
    text_embs = F.normalize(text_embs, dim=-1)

# Evaluate on first 500 images
correct, total = 0, 0
for imgs_raw, labels in sample_loader:
    imgs_raw = (imgs_raw * 255).byte().permute(0,2,3,1).numpy()
    imgs_pil = [Image.fromarray(im) for im in imgs_raw]
    inputs   = siglip_processor(images=imgs_pil, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        # (batch_num, embed_dim)
        img_feats = siglip_pretrained.get_image_features(**inputs)
        img_feats = F.normalize(img_feats, dim=-1)
    # (batch_num, num_classes)
    sims  = img_feats @ text_embs.T
    preds = sims.argmax(dim=1).cpu()
    correct += (preds == labels).sum().item()
    total   += len(labels)
    if total >= 500: break

print(f"\nPretrained SigLIP zero-shot accuracy on CIFAR-10 (500 samples): {correct/total:.3f}")
print("(MiniSigLIP trained for 5 epochs is expected to be much lower — "
      "SigLIP needed 2B pairs and 10K+ steps)")

# ── Show internal similarity matrix ───────────────────────────────────────────
imgs_s, labels_s = next(iter(DataLoader(sample_ds, batch_size=20, shuffle=True)))
imgs_pil_s = [(imgs_s[i].permute(1,2,0).numpy() * 255).astype(np.uint8) for i in range(20)]
imgs_pil_s = [Image.fromarray(im) for im in imgs_pil_s]

inputs_s = siglip_processor(images=imgs_pil_s, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    feats_s = F.normalize(siglip_pretrained.get_image_features(**inputs_s), dim=-1).cpu()

sim_s = (feats_s @ text_embs.cpu().T).numpy()  # (20, 10)

plt.figure(figsize=(9, 4))
plt.imshow(sim_s, cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
plt.xticks(range(10), CIFAR_CLASSES, rotation=45, ha="right")
plt.yticks(range(20), [CIFAR_CLASSES[l] for l in labels_s.tolist()], fontsize=7)
plt.xlabel("Class text prompt"); plt.ylabel("Image (true class)")
plt.colorbar(label="cosine similarity"); plt.title("Pretrained SigLIP — image × class text")
plt.tight_layout(); plt.show()

# Section 8 Summary — CLIP vs SigLIP

**The one equation that changed everything**

```python
# CLIP: softmax normalises over the entire batch
loss = F.cross_entropy(logits, labels)           # logits: (N, N)

# SigLIP: sigmoid is independent per pair
labels = 2*torch.eye(N) - 1                      # +1 diagonal, -1 elsewhere
loss   = -F.logsigmoid(labels * logits).mean()   # (N, N) → scalar
```

**Why this one change mattered so much**

| Property | Consequence |
|---|---|
| No batch-level normalisation | Each GPU processes its local pairs independently |
| $N^2$ binary signals | More gradient signal per batch element |
| Learnable bias $b$ | Better calibration — model knows when to abstain |
| Works at $N \leq 1024$ | Accessible to researchers without 512-GPU clusters |

**Where to look next**
- **SigLIP 2** (Tschannen et al., 2025): adds masked prediction + captioning loss on top of sigmoid contrastive — closes the gap with DINOv2 on dense tasks.
- **InternVL3 / Qwen2.5-VL**: SigLIP image encoder + dedicated VLM training recipe gives SOTA on multimodal benchmarks as of early 2025.